# Data Reorganization & Balancing Pipeline

In this notebook, we automate the transformation of our 5-class dataset into a perfectly balanced **binary dataset** (Healthy vs. Sick). 

### Why build a Pipeline?
Instead of manually copying and modifying the original dataset, we use a Python script to create a new, clean environment (`Dataset_Knee_Osteoarthritis_Balenced`). This approach is better because:
1. **Safety**: We never alter or delete the original raw data.
2. **Automation**: By using a dynamic function, we apply the exact same mathematical logic to our `train`, `test`, and `val` folders without duplicating code, which prevents bugs.

### How does the balancing work?
To avoid model bias, the number of "Sick" images must equal the number of "Healthy" images. For each directory, the script will:
- **Step 1**: Count the exact number of "Healthy" images (Grade 0).
- **Step 2**: Calculate the total number of "Sick" images (sum of Grades 1, 2, 3, and 4).
- **Step 3**: Calculate a **retention ratio** (`Total Healthy / Total Sick`).
- **Step 4**: Copy 100% of the "Healthy" images to a new `healthy` folder.
- **Step 5**: Randomly sample images from the "Sick" grades using the calculated ratio, and copy them to a new `sick` folder. 

*Note: During Step 5, we rename the sick images to include their original grade (e.g., `grade2_image.jpg`). This ensures we don't accidentally overwrite files that share the same name across different folders.*

In [1]:
import os
import shutil
import random

def balance_and_transfer(source_dir, final_dir):
    #1. Creating destination directories
    os.makedirs(os.path.join(final_dir, "healthy"), exist_ok=True)
    os.makedirs(os.path.join(final_dir, "sick"), exist_ok=True)
    
    #2. Calculating the ratio
    healthy_files = os.listdir(os.path.join(source_dir, "0"))
    total_healthy = len(healthy_files)
    
    sick_grades = ["1", "2", "3", "4"]
    total_sick_initial = 0
    for g in sick_grades:
        total_sick_initial += len(os.listdir(os.path.join(source_dir, g)))
        
    ratio = total_healthy / total_sick_initial
    print(f"- Case retention ratio: {ratio*100:.2f}%")
    
    #3. Transfer of Healthy Cases (Grade 0)
    print("- Copy of healthy images...")
    for filename in healthy_files:
        src = os.path.join(source_dir, "0", filename)
        dst = os.path.join(final_dir, "healthy", filename)
        shutil.copy(src, dst)
        
    #4. Transfer of Sick Cases (Grades 1 to 4)
    print("- Copy and balance of diseased images...")
    for g in sick_grades:
        current_grade_path = os.path.join(source_dir, g)
        all_imgs = os.listdir(current_grade_path)
        
        num_to_keep = int(len(all_imgs) * ratio)
        imgs_to_copy = random.sample(all_imgs, num_to_keep)
        
        for filename in imgs_to_copy:
            src = os.path.join(current_grade_path, filename)
            dst = os.path.join(final_dir, "sick", f"grade{g}_{filename}")
            shutil.copy(src, dst)
            
    # 5. Transfer Summary
    final_healthy_count = len(os.listdir(os.path.join(final_dir, 'healthy')))
    final_sick_count = len(os.listdir(os.path.join(final_dir, 'sick')))
    print(f"-> Success ! Healthy: {final_healthy_count} | Sick: {final_sick_count}\n")

base_source = "../Dataset_Knee_Osteoarthritis"
base_final = "../Dataset_Knee_Osteoarthritis_Balanced"

repertoires = ["train", "test", "val"]

for rep in repertoires:
    src_path = os.path.join(base_source, rep)
    dst_path = os.path.join(base_final, rep)

    balance_and_transfer(src_path, dst_path)

print("The entire database has been successfully reorganized and balanced!")


- Case retention ratio: 65.46%
- Copy of healthy images...
- Copy and balance of diseased images...
-> Success ! Healthy: 2286 | Sick: 3491

- Case retention ratio: 62.83%
- Copy of healthy images...
- Copy and balance of diseased images...
-> Success ! Healthy: 639 | Sick: 1017

- Case retention ratio: 65.86%
- Copy of healthy images...
- Copy and balance of diseased images...
-> Success ! Healthy: 328 | Sick: 498

The entire database has been successfully reorganized and balanced!
